# Bronze Layer - Ingestão dos Dados Brutos

**Data:** 27-28/06/2026  
**Objetivo:** Carregar os CSVs pra dentro do Databricks

Bom, então aqui é só a ingestão mesmo. A ideia da camada bronze é pegar os dados **exatamente como vieram** e jogar pro Delta Lake.

**O que tá fazendo:**
- Lendo 7 arquivos CSV da pasta `source/`
- Convertendo pra Delta Lake (formato otimizado)
- Salvando como tabelas gerenciadas no Unity Catalog
- Mantendo tudo como STRING (sem interpretação de tipos ainda)

**O que NÃO tá fazendo:**
- Nenhuma limpeza de dados
- Nenhuma validação
- Nenhuma transformação (isso fica pra Silver)

> 💡 Dica: Se precisar reprocessar algo, é só rodar esse notebook de novo.  
> Os dados originais em CSV ficam intactos na pasta source/

---

**Schema de saída:** `workspace.bronze.*`

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("tech-challenge-bronze").getOrCreate()

BRONZE_CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"
SOURCE_PATH = "/Workspace/Users/justinofilipe03@gmail.com/TechChallenge2/Bronze/source"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_CATALOG}.{BRONZE_SCHEMA}")
print(f"✓ Schema {BRONZE_CATALOG}.{BRONZE_SCHEMA} pronto")

✓ Schema workspace.bronze pronto


In [0]:
def normalizar_nome_coluna(nome):
    """
    Só normaliza os nomes das colunas porque Delta Lake não aceita espaços e acentos
    
    Isso aqui NÃO é tratamento de dados! É só uma limitação técnica do formato.
    O conteúdo dos dados continua exatamente como veio.
    """
    import re
    import unicodedata
    
    nome = unicodedata.normalize('NFKD', nome).encode('ASCII', 'ignore').decode('utf-8')
    nome = nome.lower()
    nome = re.sub(r'[^a-z0-9]+', '_', nome)
    nome = re.sub(r'_+', '_', nome).strip('_')
    return nome


def ingerir_csv_para_bronze(nome_arquivo, nome_tabela):
    """Lê CSV bruto e converte para Delta Lake
    
    Bronze = APENAS ingestão, SEM tratamento:
    - Lê CSV como está
    - Converte para Delta/Parquet
    - Mantém tipos originais (string)
    - Normaliza apenas nomes de colunas para compatibilidade técnica Delta
    
    TODO tratamento de dados (tipos, valores, validações) = Silver
    """
    csv_path = f"{SOURCE_PATH}/{nome_arquivo}"
    
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "false") \
        .option("encoding", "UTF-8") \
        .load(csv_path)
    
    # Normalizar nomes das colunas
    for col_name in df.columns:
        novo_nome = normalizar_nome_coluna(col_name)
        if novo_nome != col_name:
            df = df.withColumnRenamed(col_name, novo_nome)
    
    tabela_bronze = f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.{nome_tabela}"
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_bronze)
    
    count = df.count()
    print(f"✓ {nome_tabela}: {count:,} registros ingeridos")
    return df

In [0]:
# Antes de processar tudo, deixa eu ver como é a estrutura de um desses CSVs
# Vou usar o indicador_uf como exemplo

# teste_df = spark.read.format("csv") \
#     .option("header", "true") \
#     .option("inferSchema", "false") \
#     .load(f"{SOURCE_PATH}/indicador_alfabetizacao_uf.csv")

# print("Colunas encontradas:")
# for col in teste_df.columns:
#     print(f"  - {col}")

# print("\nPrimeiras linhas:")
# teste_df.show(3, truncate=False)

# Ok, parece ok. Tem alguns nomes de coluna com espaço e acento, vou ter que normalizar
# Mas o conteúdo dos dados tá legível

print("Teste exploratório comentado. Descomente se quiser rodar novamente.")

## Ingestão das Tabelas Bronze

In [0]:
# Indicadores de alfabetização
indicador_uf = ingerir_csv_para_bronze("indicador_alfabetizacao_uf.csv", "indicador_alfabetizacao_uf")
indicador_municipio = ingerir_csv_para_bronze("indicador_alfabetizacao_municipio.csv", "indicador_alfabetizacao_municipio")

# Metas de alfabetização  
meta_brasil = ingerir_csv_para_bronze("meta_alfabetizacao_brasil.csv", "meta_alfabetizacao_brasil")
meta_uf = ingerir_csv_para_bronze("meta_alfabetizacao_uf.csv", "meta_alfabetizacao_uf")
meta_municipio = ingerir_csv_para_bronze("meta_alfabetizacao_municipio.csv", "meta_alfabetizacao_municipio")

# Microdados
microdados = ingerir_csv_para_bronze("microdados_alunos.csv", "microdados_alunos")

# Resultados e metas UFs 2025
resultados_uf_2025 = ingerir_csv_para_bronze("resultados_e_metas_ufs_2025.csv", "resultados_e_metas_ufs_2025")

✓ indicador_alfabetizacao_uf: 145 registros ingeridos
✓ indicador_alfabetizacao_municipio: 23,995 registros ingeridos
✓ meta_alfabetizacao_brasil: 3 registros ingeridos
✓ meta_alfabetizacao_uf: 54 registros ingeridos
✓ meta_alfabetizacao_municipio: 10,704 registros ingeridos
✓ microdados_alunos: 3,867,999 registros ingeridos
✓ resultados_e_metas_ufs_2025: 117 registros ingeridos


## Validação Final

In [0]:
tables = spark.sql(f"SHOW TABLES IN {BRONZE_CATALOG}.{BRONZE_SCHEMA}").collect()
print(f"\n=== Tabelas Bronze Criadas ({len(tables)}) ===")
for table in tables:
    count = spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.{table.tableName}").count()
    print(f"  • {table.tableName}: {count:,} registros")


=== Tabelas Bronze Criadas (7) ===
  • indicador_alfabetizacao_municipio: 23,995 registros
  • indicador_alfabetizacao_uf: 145 registros
  • meta_alfabetizacao_brasil: 3 registros
  • meta_alfabetizacao_municipio: 10,704 registros
  • meta_alfabetizacao_uf: 54 registros
  • microdados_alunos: 3,867,999 registros
  • resultados_e_metas_ufs_2025: 117 registros
